# Private LoRA4 final benchmark evaluation
Use free T4 GPU and Internet. Attach private original Task1 data and pa3-lora4-final-eval-input.zip. No training occurs. Runs full HellaSwag, BoolQ, ARC-Easy and GSM8K with the existing fixed protocol. Outputs contain actual results and generation caches; partial completion is explicitly recorded. Do not overlap another GPU job unless quota and resources are independently confirmed.


In [ ]:
# Set only after the previous Kaggle draft is stopped and no training is active.
BATCH_HANDOFF_CONFIRMED = False
assert BATCH_HANDOFF_CONFIRMED, 'Stop the prior draft/session, then confirm handoff before Save & Run All'


In [ ]:
import subprocess, sys, pathlib, json, zipfile, urllib.request
REPO_COMMIT = "2bf48e2b3bb9131a68941ba1eed05ae8bee5a475"
work = pathlib.Path("/kaggle/working")
source = work / "pa3-source"
if not source.exists():
    archive = work / "pa3-source.zip"
    urllib.request.urlretrieve(f"https://github.com/sohauv/atml-pa3/archive/{REPO_COMMIT}.zip", archive)
    with zipfile.ZipFile(archive) as z:
        z.extractall(work)
    (work / f"atml-pa3-{REPO_COMMIT}").rename(source)
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "torch==2.6.0", "torchvision==0.21.0", "torchaudio==2.6.0", "transformers==4.51.3", "accelerate==1.6.0", "datasets==3.5.0", "numpy==2.2.5"])
print("Pinned source prepared:", REPO_COMMIT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'lm_eval==0.4.8', 'peft==0.15.2', 'math-verify[antlr4_13_2]==0.8.0'])


In [ ]:
# CPU-only input verification; accept Kaggle automatic ZIP extraction.
import sys, pathlib, zipfile, hashlib, json
sys.path.insert(0, str(source))
from scripts.kaggle_task1 import unpack_inputs
archives = list(pathlib.Path("/kaggle/input").rglob("pa3-task1-input.zip"))
if len(archives) == 1:
    input_archive = archives[0]
else:
    candidates = [p.parent for p in pathlib.Path("/kaggle/input").rglob("metadata.json") if (p.parent / "train.jsonl").is_file() and (p.parent / "source_ids.json").is_file()]
    if len(candidates) != 1:
        raise RuntimeError(f"Expected one original Task1 input; found {len(candidates)}")
    original = candidates[0]
    actual = hashlib.sha256((original / "train.jsonl").read_bytes()).hexdigest()
    if actual != json.loads((original / "metadata.json").read_text())["sha256"]:
        raise RuntimeError("Original training checksum mismatch")
    input_archive = work / "pa3-task1-input.zip"
    if not input_archive.exists():
        with zipfile.ZipFile(input_archive, "x", zipfile.ZIP_DEFLATED) as z:
            for name in ("train.jsonl", "metadata.json", "source_ids.json"):
                z.write(original / name, "task1/" + name)
data_dir = unpack_inputs(input_archive, work / "pa3/data")
print("Original training data checksum verified:", data_dir)


In [ ]:
import pathlib, json, shutil, hashlib
EXPECTED = {'lora4-recovery/step-1250/projected_factors.pt': '5194fba34c17f43aed1b6c0fb3488009cb1839fb61f6f86d38ca7502d33e7a05', 'lora4-recovery/step-1250/complete.json': '1a72b933ab0ce3640086315afa84306682bbe5768a8e24ef5f7c9a328c58e88d', 'lora4-recovery/step-1250/recovery.pt': 'b82b9626f8da6893d956b43a605ed965ba77de9aebad94b0ce3c8cbf6df35198', 'lora4-recovery/step-1250/summary.json': '20478b410663ad502d356dd33f1a1f164facb3ad78555af10d2d7a276e6b5098'}
candidates = [p.parent for p in pathlib.Path('/kaggle/input').rglob('complete.json') if p.parent.name == 'step-1250' and (p.parent / 'projected_factors.pt').is_file()]
assert len(candidates) == 1, 'Attach exactly one private final LoRA4 evaluation checkpoint'
original = candidates[0]
checkpoint = work / 'lora4-eval-input/step-1250'
assert not checkpoint.exists(), 'Fresh saved evaluation job required'
for name, digest in EXPECTED.items():
    file = original / pathlib.PurePosixPath(name).name
    assert file.is_file() and hashlib.file_digest(file.open('rb'), 'sha256').hexdigest() == digest, str(file)
shutil.copytree(original, checkpoint)
summary = json.loads((checkpoint / 'summary.json').read_text())
assert summary['update'] == 1250 and summary['successful_optimizer_updates'] == 1250 and summary['skipped_updates'] == 0
print('VERIFIED_FINAL_LORA4_FOR_EVALUATION', summary, flush=True)


In [ ]:
import subprocess, sys, pathlib, zipfile, hashlib, json
output = work / 'lora4-final-evaluation'
command = [sys.executable, '-m', 'scripts.kaggle_evaluate_task1', '--data-dir', str(data_dir), '--checkpoint', str(checkpoint), '--output', str(output), '--seconds', '18000']
print('START_FULL_LORA4_BENCHMARKS', command, flush=True)
result = subprocess.run(command, cwd=source)
completed = [task for task in ['hellaswag','boolq','arc_easy','gsm8k_cot'] if (output / (task+'.json')).is_file()]
evidence = {'returncode':result.returncode, 'completed_tasks':completed, 'all_four_complete':len(completed)==4, 'checkpoint_update':1250, 'smoke_limit':None}
(work / 'lora4-evaluation-status.json').write_text(json.dumps(evidence,indent=2))
archive = work / 'lora4-final-evaluation-results.zip'
with zipfile.ZipFile(archive, 'x', zipfile.ZIP_DEFLATED) as z:
    if output.exists():
        for p in output.rglob('*'):
            if p.is_file(): z.write(p,str(pathlib.Path('lora4-final-evaluation') / p.relative_to(output)))
    z.write(work / 'lora4-evaluation-status.json','lora4-evaluation-status.json')
with zipfile.ZipFile(archive) as z: assert z.testzip() is None
print('ACTUAL_EVALUATION_STATUS', json.dumps(evidence), flush=True)
print('DOWNLOAD_RESULTS', archive, archive.stat().st_size, hashlib.file_digest(archive.open('rb'),'sha256').hexdigest(), flush=True)
if not evidence['all_four_complete']: print('INCOMPLETE_EVALUATION: preserve output/cache and inspect failure before any continuation', flush=True)
